# Lab A — Contract Intelligence Foundations

## Architecture

```text
Synthetic enterprise sources
    │
    ├── Supplier master
    ├── Ariba metadata
    ├── ERP payment schedule
    ├── SharePoint / repository metadata
    └── Contract documents
            │
            ▼
     AI structured extraction
            │
            ├── obligations
            ├── milestones
            ├── payments
            └── risks
            │
            ▼
     Normalized Delta tables
            │
            ├──────────────────────────────┐
            │                              │
            ▼                              ▼
 Enterprise Contract View          Clause-level chunks
            │                              │
            │                              ▼
            │                     Retrieval / AI Search-ready
            │                              │
            └──────────────┬───────────────┘
                           ▼
                 Agent-ready foundation
                           │
             ┌─────────────┴─────────────┐
             ▼                           ▼
 query_contract_data(...)   retrieve_contract_evidence(...)
```

### POC objective
Create a small enterprise contract intelligence foundation using synthetic data so a later agentic workflow can reason across both structured contract facts and unstructured contractual evidence.

### Main outputs
- Supplier and contract master data
- Ariba metadata
- ERP payment schedule
- Contract documents
- AI-extracted obligations, milestones, payments, and risks
- Clause-level retrieval corpus
- Consolidated enterprise contract view
- Structured query tool
- Contract evidence retrieval tool

### Free Edition approach
- Serverless-compatible DataFrame and SQL operations
- `ai_query` attempted for extraction
- Deterministic fallback if model access is unavailable
- Local TF-IDF retrieval available immediately
- Clause Delta table prepared for optional Databricks AI Search
- Materialized view attempted with standard-view fallback


## Section 1 — Configure the POC

In [0]:
# Discover the active catalog and schema and define reusable POC object names.
row = spark.sql("SELECT current_catalog() AS catalog_name, current_schema() AS schema_name").first()

CATALOG = row["catalog_name"]
SCHEMA = row["schema_name"]
PREFIX = "contract_intel_poc"

SUPPLIER_TABLE = f"{CATALOG}.{SCHEMA}.{PREFIX}_suppliers"
CONTRACT_TABLE = f"{CATALOG}.{SCHEMA}.{PREFIX}_contracts"
DOCUMENT_TABLE = f"{CATALOG}.{SCHEMA}.{PREFIX}_documents"
ARIBA_TABLE = f"{CATALOG}.{SCHEMA}.{PREFIX}_ariba"
ERP_TABLE = f"{CATALOG}.{SCHEMA}.{PREFIX}_erp_payments"
SHAREPOINT_TABLE = f"{CATALOG}.{SCHEMA}.{PREFIX}_sharepoint"
EXTRACTION_TABLE = f"{CATALOG}.{SCHEMA}.{PREFIX}_extractions"
OBLIGATION_TABLE = f"{CATALOG}.{SCHEMA}.{PREFIX}_obligations"
MILESTONE_TABLE = f"{CATALOG}.{SCHEMA}.{PREFIX}_milestones"
PAYMENT_TABLE = f"{CATALOG}.{SCHEMA}.{PREFIX}_payments"
RISK_TABLE = f"{CATALOG}.{SCHEMA}.{PREFIX}_risks"
CLAUSE_TABLE = f"{CATALOG}.{SCHEMA}.{PREFIX}_clauses"
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.{PREFIX}_enterprise_gold"
VIEW_NAME = f"{CATALOG}.{SCHEMA}.{PREFIX}_enterprise_contract_view"

LLM_ENDPOINT = "system.ai.meta-llama-3-3-70b-instruct"
RUN_LLM_EXTRACTION = True
RUN_AI_SEARCH_SETUP = False

print("Catalog:", CATALOG)
print("Schema:", SCHEMA)


## Section 2 — Generate Synthetic Enterprise Sources

In [0]:
# Create synthetic supplier master data and persist it as a Delta table.
import pandas as pd
import numpy as np
import random
from datetime import date, timedelta

random.seed(42)
np.random.seed(42)

suppliers = [
    ("SUP001", "Apex Industrial Systems", "Industrial Equipment", "India", "Strategic"),
    ("SUP002", "BluePeak Technology Services", "IT Services", "India", "Strategic"),
    ("SUP003", "Crestline Logistics", "Logistics", "Singapore", "Preferred"),
    ("SUP004", "Delta Process Automation", "Automation", "Germany", "Strategic"),
    ("SUP005", "Evergreen Facilities", "Facilities", "India", "Preferred"),
    ("SUP006", "Fusion Cloud Solutions", "Cloud Services", "USA", "Strategic"),
]

supplier_pdf = pd.DataFrame(
    suppliers,
    columns=["supplier_id", "supplier_name", "category", "country", "supplier_tier"]
)

supplier_df = spark.createDataFrame(supplier_pdf)
supplier_df.write.mode("overwrite").format("delta").saveAsTable(SUPPLIER_TABLE)

display(supplier_df)


In [0]:
# Generate contract, Ariba, ERP payment, SharePoint, and synthetic ground-truth records.
base_date = date(2026, 9, 1)
contract_rows, ariba_rows, erp_rows, sharepoint_rows, truth_rows = [], [], [], [], []

for i in range(1, 13):
    supplier_id, supplier_name, _, _, _ = suppliers[(i - 1) % len(suppliers)]
    contract_id = f"CTR-2026-{i:03d}"
    ariba_id = f"ARIBA-{8000+i}"
    start_date = base_date - timedelta(days=180 + i * 7)
    end_date = base_date + timedelta(days=90 + i * 25)
    contract_value = float(4_000_000 + i * 1_350_000)
    currency = ["INR", "USD", "EUR"][i % 3]
    owner = ["Procurement", "Finance", "IT", "Operations"][i % 4]

    m1_date = base_date + timedelta(days=10 + i * 3)
    m2_date = base_date + timedelta(days=40 + i * 4)
    m1 = f"Complete design and readiness review for work package {i}"
    m2 = f"Complete production handover for work package {i}"

    p1 = round(contract_value * 0.35, 2)
    p2 = round(contract_value * 0.45, 2)
    p3 = round(contract_value * 0.20, 2)
    p1_date = m1_date + timedelta(days=15)
    p2_date = m2_date + timedelta(days=20)
    p3_date = end_date - timedelta(days=15)

    severity = ["LOW", "MEDIUM", "HIGH"][i % 3]
    risk_type = ["Delivery Delay", "SLA Exposure", "Renewal Risk", "Payment Dependency"][i % 4]
    sla = 97 + (i % 3)
    penalty = round(0.25 + (i % 4) * 0.15, 2)
    renewal_days = [30, 60, 90][i % 3]
    commitment = f"Deliver contracted services and milestone outputs for work package {i}."
    action = {
        "LOW": "Track through normal contract governance.",
        "MEDIUM": "Request supplier status update and confirm owner action.",
        "HIGH": "Escalate to procurement owner and initiate mitigation review."
    }[severity]

    contract_rows.append((
        contract_id, supplier_id, ariba_id, ["Ariba", "SharePoint", "Contract Repository"][i % 3],
        start_date.isoformat(), end_date.isoformat(), contract_value, currency, owner, "ACTIVE"
    ))

    ariba_rows.append((ariba_id, contract_id, supplier_id, f"PO-{50000+i}", owner, "Approved"))

    for n, milestone, amount, due_date in [
        (1, m1, p1, p1_date),
        (2, m2, p2, p2_date),
        (3, "Final acceptance", p3, p3_date),
    ]:
        erp_rows.append((contract_id, n, milestone, float(amount), due_date.isoformat(), "OPEN"))

    sharepoint_rows.append((
        contract_id, f"/Contracts/2026/{contract_id}.pdf", f"{contract_id}.pdf",
        (base_date - timedelta(days=i)).isoformat(), "Legal-Procurement"
    ))

    truth_rows.append({
        "contract_id": contract_id,
        "supplier_name": supplier_name,
        "commitment": commitment,
        "milestone_1": m1,
        "milestone_1_date": m1_date.isoformat(),
        "milestone_2": m2,
        "milestone_2_date": m2_date.isoformat(),
        "payment_1": p1,
        "payment_1_date": p1_date.isoformat(),
        "payment_2": p2,
        "payment_2_date": p2_date.isoformat(),
        "payment_3": p3,
        "payment_3_date": p3_date.isoformat(),
        "currency": currency,
        "sla": sla,
        "penalty": penalty,
        "renewal_days": renewal_days,
        "risk_type": risk_type,
        "risk_severity": severity,
        "recommended_action": action
    })

spark.createDataFrame(contract_rows, [
    "contract_id", "supplier_id", "ariba_contract_id", "source_system",
    "start_date", "end_date", "contract_value", "currency", "contract_owner", "status"
]).write.mode("overwrite").format("delta").saveAsTable(CONTRACT_TABLE)

spark.createDataFrame(ariba_rows, [
    "ariba_contract_id", "contract_id", "supplier_id", "purchase_order", "business_owner", "approval_status"
]).write.mode("overwrite").format("delta").saveAsTable(ARIBA_TABLE)

spark.createDataFrame(erp_rows, [
    "contract_id", "payment_no", "linked_milestone", "amount", "due_date", "payment_status"
]).write.mode("overwrite").format("delta").saveAsTable(ERP_TABLE)

spark.createDataFrame(sharepoint_rows, [
    "contract_id", "document_path", "file_name", "last_modified", "security_group"
]).write.mode("overwrite").format("delta").saveAsTable(SHAREPOINT_TABLE)

truth_pdf = pd.DataFrame(truth_rows)

display(spark.table(CONTRACT_TABLE))


In [0]:
# Generate realistic synthetic contract text and persist the document corpus.
document_rows = []

for t in truth_rows:
    meta = next(r for r in contract_rows if r[0] == t["contract_id"])

    text = f"""
SECTION 1 - CONTRACT OVERVIEW
Contract ID: {t['contract_id']}
Supplier: {t['supplier_name']}
Contract value: {meta[6]:,.2f} {meta[7]}
Effective date: {meta[4]}
Expiry date: {meta[5]}

SECTION 2 - SUPPLIER COMMITMENTS
The Supplier commits to: {t['commitment']}
The supplier must maintain required staffing, delivery capacity, governance reporting, and evidence of completion.

SECTION 3 - MILESTONES
Milestone 1: {t['milestone_1']}. Target date: {t['milestone_1_date']}.
Milestone 2: {t['milestone_2']}. Target date: {t['milestone_2_date']}.
Final acceptance is required before contract closure.

SECTION 4 - PAYMENT SCHEDULE
Payment 1: {t['payment_1']:,.2f} {t['currency']} due on {t['payment_1_date']} after milestone acceptance.
Payment 2: {t['payment_2']:,.2f} {t['currency']} due on {t['payment_2_date']} after production handover.
Payment 3: {t['payment_3']:,.2f} {t['currency']} due on {t['payment_3_date']} after final acceptance.
Payments may be placed on hold when linked milestone evidence is incomplete.

SECTION 5 - SERVICE LEVEL AND PENALTIES
The Supplier must maintain an SLA attainment of at least {t['sla']} percent.
Delay beyond the agreed milestone date may attract a penalty of {t['penalty']} percent of the affected milestone value per week.

SECTION 6 - RENEWAL AND TERMINATION
The contract requires renewal review at least {t['renewal_days']} days before expiry.
Material breach, repeated SLA failure, or non-delivery may trigger termination review.

SECTION 7 - IDENTIFIED RISK
Risk type: {t['risk_type']}.
Risk severity: {t['risk_severity']}.
Recommended action: {t['recommended_action']}
""".strip()

    document_rows.append((t["contract_id"], f"{t['contract_id']}.pdf", text))

document_df = spark.createDataFrame(
    document_rows,
    ["contract_id", "document_name", "contract_text"]
)

document_df.write.mode("overwrite").format("delta").saveAsTable(DOCUMENT_TABLE)

display(document_df.limit(3))


## Section 3 — AI Structured Extraction

In [0]:
# Define a strict LLM extraction prompt and a deterministic fallback for unavailable model access.
import json
import re

EXTRACTION_PROMPT = """
You are an enterprise contract intelligence extraction engine.
Return ONLY valid JSON with this structure:
{
  "supplier_name": "string",
  "commitment": "string",
  "milestones": [{"milestone": "string", "target_date": "YYYY-MM-DD"}],
  "payments": [{"payment_no": 1, "linked_milestone": "string", "amount": 0.0, "currency": "string", "due_date": "YYYY-MM-DD"}],
  "sla_target_pct": 0.0,
  "delay_penalty_pct_per_week": 0.0,
  "renewal_notice_days": 0,
  "risk_type": "string",
  "risk_severity": "LOW|MEDIUM|HIGH",
  "recommended_action": "string"
}
CONTRACT:
"""

def fallback_extract(contract_id):
    t = truth_pdf.loc[truth_pdf["contract_id"] == contract_id].iloc[0].to_dict()
    return {
        "supplier_name": t["supplier_name"],
        "commitment": t["commitment"],
        "milestones": [
            {"milestone": t["milestone_1"], "target_date": t["milestone_1_date"]},
            {"milestone": t["milestone_2"], "target_date": t["milestone_2_date"]}
        ],
        "payments": [
            {"payment_no": 1, "linked_milestone": t["milestone_1"], "amount": t["payment_1"], "currency": t["currency"], "due_date": t["payment_1_date"]},
            {"payment_no": 2, "linked_milestone": t["milestone_2"], "amount": t["payment_2"], "currency": t["currency"], "due_date": t["payment_2_date"]},
            {"payment_no": 3, "linked_milestone": "Final acceptance", "amount": t["payment_3"], "currency": t["currency"], "due_date": t["payment_3_date"]}
        ],
        "sla_target_pct": float(t["sla"]),
        "delay_penalty_pct_per_week": float(t["penalty"]),
        "renewal_notice_days": int(t["renewal_days"]),
        "risk_type": t["risk_type"],
        "risk_severity": t["risk_severity"],
        "recommended_action": t["recommended_action"]
    }


In [0]:
# Attempt ai_query extraction for every contract and fall back automatically if the system model cannot be used.
def clean_json_text(value):
    value = (value or "").strip()
    value = re.sub(r"^```json\s*", "", value, flags=re.I)
    value = re.sub(r"^```\s*", "", value)
    value = re.sub(r"\s*```$", "", value)
    return value.strip()

extractions = []
EXTRACTION_MODE = "deterministic_fallback"

if RUN_LLM_EXTRACTION:
    try:
        for row in spark.table(DOCUMENT_TABLE).collect():
            prompt = (EXTRACTION_PROMPT + "\n" + row["contract_text"]).replace("'", "''")
            endpoint = LLM_ENDPOINT.replace("'", "''")

            response = spark.sql(
                f"SELECT ai_query('{endpoint}', '{prompt}') AS response"
            ).first()["response"]

            parsed = json.loads(clean_json_text(response))
            parsed["contract_id"] = row["contract_id"]
            parsed["extraction_method"] = "ai_query"
            extractions.append(parsed)

        EXTRACTION_MODE = "ai_query"

    except Exception as e:
        print("ai_query unavailable; deterministic fallback will be used.")
        print(type(e).__name__, str(e)[:500])
        extractions = []

if not extractions:
    for contract_id in truth_pdf["contract_id"].tolist():
        parsed = fallback_extract(contract_id)
        parsed["contract_id"] = contract_id
        parsed["extraction_method"] = "deterministic_fallback"
        extractions.append(parsed)

print("Extraction mode:", EXTRACTION_MODE)
print("Contracts extracted:", len(extractions))


In [0]:
# Persist raw extraction JSON so the AI output is auditable before normalization.
extraction_df = spark.createDataFrame(
    [
        (x["contract_id"], x["extraction_method"], json.dumps(x))
        for x in extractions
    ],
    ["contract_id", "extraction_method", "extraction_json"]
)

extraction_df.write.mode("overwrite").format("delta").saveAsTable(EXTRACTION_TABLE)

display(extraction_df.limit(5))


## Section 4 — Normalize Contract Intelligence

In [0]:
# Normalize extracted obligations, milestones, payments, and risks into separate governed Delta tables.
obligations, milestones, payments, risks = [], [], [], []

for x in extractions:
    cid = x["contract_id"]
    obligations.append((cid, "SUPPLIER_COMMITMENT", x.get("commitment"), "OPEN"))

    for n, m in enumerate(x.get("milestones", []), start=1):
        milestones.append((cid, n, m.get("milestone"), m.get("target_date"), "OPEN"))

    for p in x.get("payments", []):
        payments.append((
            cid,
            int(p.get("payment_no", 0)),
            p.get("linked_milestone"),
            float(p.get("amount", 0.0)),
            p.get("currency"),
            p.get("due_date"),
            "OPEN"
        ))

    risks.append((
        cid,
        x.get("risk_type"),
        x.get("risk_severity"),
        x.get("recommended_action"),
        float(x.get("sla_target_pct", 0.0)),
        float(x.get("delay_penalty_pct_per_week", 0.0)),
        int(x.get("renewal_notice_days", 0))
    ))

spark.createDataFrame(obligations, [
    "contract_id", "obligation_type", "obligation_text", "status"
]).write.mode("overwrite").format("delta").saveAsTable(OBLIGATION_TABLE)

spark.createDataFrame(milestones, [
    "contract_id", "milestone_no", "milestone", "target_date", "status"
]).write.mode("overwrite").format("delta").saveAsTable(MILESTONE_TABLE)

spark.createDataFrame(payments, [
    "contract_id", "payment_no", "linked_milestone", "amount", "currency", "due_date", "payment_status"
]).write.mode("overwrite").format("delta").saveAsTable(PAYMENT_TABLE)

spark.createDataFrame(risks, [
    "contract_id", "risk_type", "risk_severity", "recommended_action",
    "sla_target_pct", "delay_penalty_pct_per_week", "renewal_notice_days"
]).write.mode("overwrite").format("delta").saveAsTable(RISK_TABLE)

display(spark.table(RISK_TABLE))


In [0]:
# Run compact data-quality checks and compare key extracted fields with the synthetic ground truth.
from pyspark.sql import functions as F

quality_rows = [
    ("duplicate_contract_ids", spark.table(CONTRACT_TABLE).groupBy("contract_id").count().filter(F.col("count") > 1).count()),
    ("invalid_payment_amounts", spark.table(PAYMENT_TABLE).filter(F.col("amount") <= 0).count()),
    ("missing_milestone_dates", spark.table(MILESTONE_TABLE).filter(F.col("target_date").isNull()).count()),
    ("missing_risk_severity", spark.table(RISK_TABLE).filter(F.col("risk_severity").isNull()).count()),
]

quality_df = spark.createDataFrame(
    [(name, issues, issues == 0) for name, issues in quality_rows],
    ["check_name", "issue_count", "passed"]
)

display(quality_df)

eval_rows = []
for x in extractions:
    t = truth_pdf.loc[truth_pdf["contract_id"] == x["contract_id"]].iloc[0]
    m = x.get("milestones", [{}])[0] if x.get("milestones") else {}
    p = x.get("payments", [{}])[0] if x.get("payments") else {}

    eval_rows.append((
        x["contract_id"],
        x.get("supplier_name") == t["supplier_name"],
        m.get("target_date") == t["milestone_1_date"],
        abs(float(p.get("amount", 0.0)) - float(t["payment_1"])) < 0.01,
        x.get("risk_severity") == t["risk_severity"]
    ))

evaluation_df = spark.createDataFrame(
    eval_rows,
    ["contract_id", "supplier_correct", "milestone_date_correct", "payment_correct", "risk_correct"]
)

display(evaluation_df)


## Section 5 — Create Clause Chunks for Retrieval

In [0]:
# Split each contract into business-clause chunks and write an AI Search-ready Delta source table.
section_pattern = re.compile(
    r"SECTION\s+(\d+)\s+-\s+([^\n]+)\n(.*?)(?=\nSECTION\s+\d+\s+-|\Z)",
    re.S | re.I
)

clause_rows = []

for contract_id, document_name, contract_text in document_rows:
    for section_number, section_title, section_body in section_pattern.findall(contract_text):
        clause_rows.append((
            f"{contract_id}-S{int(section_number):02d}",
            contract_id,
            int(section_number),
            section_title.strip().upper(),
            section_body.strip(),
            document_name
        ))

clause_df = spark.createDataFrame(
    clause_rows,
    ["chunk_id", "contract_id", "section_number", "clause_type", "clause_text", "document_name"]
)

clause_df.write.mode("overwrite").format("delta").option(
    "delta.enableChangeDataFeed", "true"
).saveAsTable(CLAUSE_TABLE)

display(clause_df.limit(20))


## Section 6 — Build the Enterprise Contract View

In [0]:
# Build the consolidated Gold table with next milestone, next payment, obligation count, risks, and renewal exposure.
gold_sql = f"""
CREATE OR REPLACE TABLE {GOLD_TABLE}
USING DELTA
AS
WITH next_milestone AS (
    SELECT
        contract_id,
        milestone,
        CAST(target_date AS DATE) AS target_date,
        ROW_NUMBER() OVER (
            PARTITION BY contract_id
            ORDER BY CAST(target_date AS DATE)
        ) AS rn
    FROM {MILESTONE_TABLE}
    WHERE status = 'OPEN'
),
next_payment AS (
    SELECT
        contract_id,
        linked_milestone,
        amount,
        currency,
        CAST(due_date AS DATE) AS due_date,
        ROW_NUMBER() OVER (
            PARTITION BY contract_id
            ORDER BY CAST(due_date AS DATE)
        ) AS rn
    FROM {PAYMENT_TABLE}
    WHERE payment_status = 'OPEN'
),
obligation_count AS (
    SELECT
        contract_id,
        COUNT(*) AS open_obligations
    FROM {OBLIGATION_TABLE}
    WHERE status = 'OPEN'
    GROUP BY contract_id
)
SELECT
    c.contract_id,
    c.supplier_id,
    s.supplier_name,
    s.category,
    s.country,
    s.supplier_tier,
    c.ariba_contract_id,
    a.purchase_order,
    c.contract_owner,
    CAST(c.start_date AS DATE) AS start_date,
    CAST(c.end_date AS DATE) AS end_date,
    c.contract_value,
    c.currency,
    c.status,
    nm.milestone AS next_milestone,
    nm.target_date AS next_milestone_date,
    np.linked_milestone AS next_payment_dependency,
    np.amount AS next_payment_amount,
    np.currency AS next_payment_currency,
    np.due_date AS next_payment_due_date,
    COALESCE(oc.open_obligations, 0) AS open_obligations,
    r.risk_type,
    r.risk_severity,
    r.recommended_action,
    r.sla_target_pct,
    r.delay_penalty_pct_per_week,
    r.renewal_notice_days,
    DATEDIFF(CAST(c.end_date AS DATE), CURRENT_DATE()) AS days_to_expiry
FROM {CONTRACT_TABLE} c
JOIN {SUPPLIER_TABLE} s
    ON c.supplier_id = s.supplier_id
LEFT JOIN {ARIBA_TABLE} a
    ON c.contract_id = a.contract_id
LEFT JOIN next_milestone nm
    ON c.contract_id = nm.contract_id AND nm.rn = 1
LEFT JOIN next_payment np
    ON c.contract_id = np.contract_id AND np.rn = 1
LEFT JOIN obligation_count oc
    ON c.contract_id = oc.contract_id
LEFT JOIN {RISK_TABLE} r
    ON c.contract_id = r.contract_id
"""

spark.sql(gold_sql)

display(spark.table(GOLD_TABLE))


In [0]:
# Attempt a standalone materialized view and fall back to a standard SQL view when required.
MATERIALIZED_VIEW_CREATED = False

try:
    mv_sql = f"""
    CREATE OR REPLACE MATERIALIZED VIEW {VIEW_NAME}
    AS
    SELECT *
    FROM {GOLD_TABLE}
    """
    spark.sql(mv_sql)
    MATERIALIZED_VIEW_CREATED = True
    print("Materialized view created:", VIEW_NAME)

except Exception as e:
    print("Materialized view unavailable; creating a standard view.")
    print(type(e).__name__, str(e)[:400])

    try:
        spark.sql(f"DROP VIEW IF EXISTS {VIEW_NAME}")
    except Exception:
        pass

    view_sql = f"""
    CREATE VIEW {VIEW_NAME}
    AS
    SELECT *
    FROM {GOLD_TABLE}
    """
    spark.sql(view_sql)

display(spark.table(VIEW_NAME).limit(20))


## Section 7 — Structured Contract Intelligence Queries

In [0]:
# Demonstrate structured business questions that should use the enterprise view instead of RAG.
print("HIGH-RISK CONTRACTS")
display(
    spark.table(VIEW_NAME)
    .filter(F.col("risk_severity") == "HIGH")
    .select(
        "contract_id",
        "supplier_name",
        "next_milestone",
        "next_milestone_date",
        "next_payment_amount",
        "risk_type",
        "recommended_action"
    )
)

print("UPCOMING EXPIRY")
display(
    spark.table(VIEW_NAME)
    .filter(
        (F.col("days_to_expiry") >= 0)
        & (F.col("days_to_expiry") <= 180)
    )
    .select(
        "contract_id",
        "supplier_name",
        "end_date",
        "days_to_expiry",
        "renewal_notice_days",
        "risk_severity"
    )
    .orderBy("days_to_expiry")
)


## Section 8 — Build the Retrieval Layer

In [0]:
# Build a local TF-IDF index so clause-level contract evidence can be retrieved immediately.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

clause_pdf = spark.table(CLAUSE_TABLE).toPandas()

retrieval_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

clause_matrix = retrieval_vectorizer.fit_transform(
    clause_pdf["clause_text"].fillna("")
)

print("Indexed chunks:", len(clause_pdf))


In [0]:
# Define the reusable retrieve_contract_evidence function for agent-ready clause retrieval.
def retrieve_contract_evidence(query, contract_id=None, top_k=5):
    candidate_pdf = clause_pdf.copy()

    if contract_id is not None:
        candidate_pdf = candidate_pdf[
            candidate_pdf["contract_id"] == contract_id
        ].copy()

    if candidate_pdf.empty:
        return pd.DataFrame()

    indices = candidate_pdf.index.to_list()
    query_vector = retrieval_vectorizer.transform([query])
    scores = cosine_similarity(
        query_vector,
        clause_matrix[indices]
    ).flatten()

    result_pdf = candidate_pdf.copy()
    result_pdf["score"] = scores

    return (
        result_pdf[
            ["score", "chunk_id", "contract_id", "clause_type", "clause_text"]
        ]
        .sort_values("score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

display(
    retrieve_contract_evidence(
        "What penalty applies when a milestone is delayed?",
        top_k=5
    )
)


## Section 9 — Optional Databricks AI Search Setup

The clause table is already suitable as a Delta Sync source:
- unique `chunk_id`
- `clause_text`
- contract metadata
- Change Data Feed enabled

Set `RUN_AI_SEARCH_SETUP = True` only when you want to create the endpoint and index.


In [0]:
# Optionally create one AI Search endpoint and a Delta Sync index over the clause corpus.
AI_SEARCH_ENDPOINT_NAME = "contract-intelligence-search"
AI_SEARCH_INDEX_NAME = f"{CATALOG}.{SCHEMA}.{PREFIX}_clauses_index"
EMBEDDING_MODEL_ENDPOINT = "databricks-qwen3-embedding-0-6b"

if RUN_AI_SEARCH_SETUP:
    try:
        from databricks.ai_search.client import AISearchClient

        client = AISearchClient()

        try:
            client.get_endpoint(name=AI_SEARCH_ENDPOINT_NAME)
        except Exception:
            client.create_endpoint(
                name=AI_SEARCH_ENDPOINT_NAME,
                endpoint_type="STANDARD"
            )

        try:
            index = client.get_index(
                endpoint_name=AI_SEARCH_ENDPOINT_NAME,
                index_name=AI_SEARCH_INDEX_NAME
            )
        except Exception:
            index = client.create_delta_sync_index(
                endpoint_name=AI_SEARCH_ENDPOINT_NAME,
                source_table_name=CLAUSE_TABLE,
                index_name=AI_SEARCH_INDEX_NAME,
                pipeline_type="TRIGGERED",
                primary_key="chunk_id",
                embedding_source_column="clause_text",
                embedding_model_endpoint_name=EMBEDDING_MODEL_ENDPOINT,
                columns_to_sync=[
                    "contract_id",
                    "section_number",
                    "clause_type",
                    "clause_text",
                    "document_name"
                ]
            )

        print("AI Search setup requested.")

    except Exception as e:
        print("AI Search setup skipped or unavailable.")
        print(type(e).__name__, str(e)[:500])

else:
    print("AI Search setup disabled.")


## Section 10 — Create the Structured Query Tool

In [0]:
# Define the reusable query_contract_data function for controlled structured enterprise contract queries.
def query_contract_data(intent, days=90, supplier_id=None):
    df = spark.table(VIEW_NAME)

    if supplier_id is not None:
        df = df.filter(F.col("supplier_id") == supplier_id)

    if intent == "high_risk":
        return (
            df.filter(F.col("risk_severity") == "HIGH")
            .orderBy("next_milestone_date")
        )

    if intent == "expiring":
        return (
            df.filter(
                (F.col("days_to_expiry") >= 0)
                & (F.col("days_to_expiry") <= int(days))
            )
            .orderBy("days_to_expiry")
        )

    if intent == "payments_due":
        return (
            df.filter(F.col("next_payment_due_date").isNotNull())
            .filter(
                F.col("next_payment_due_date")
                <= F.date_add(F.current_date(), int(days))
            )
            .orderBy("next_payment_due_date")
        )

    if intent == "supplier_summary":
        return (
            df.groupBy("supplier_id", "supplier_name")
            .agg(
                F.countDistinct("contract_id").alias("contracts"),
                F.sum("contract_value").alias("total_contract_value"),
                F.sum(
                    F.when(
                        F.col("risk_severity") == "HIGH",
                        1
                    ).otherwise(0)
                ).alias("high_risk_contracts"),
                F.sum("open_obligations").alias("open_obligations")
            )
            .orderBy(F.col("total_contract_value").desc())
        )

    raise ValueError(
        "Supported intents: high_risk, expiring, payments_due, supplier_summary"
    )

display(query_contract_data("supplier_summary"))


## Section 11 — Demonstrate Hybrid Contract Intelligence

In [0]:
# Combine structured high-risk contract facts with retrieved contractual evidence in one hybrid result.
high_risk_pdf = (
    query_contract_data("high_risk")
    .select(
        "contract_id",
        "supplier_name",
        "next_milestone",
        "next_milestone_date",
        "next_payment_amount",
        "risk_type",
        "recommended_action"
    )
    .toPandas()
)

hybrid_rows = []

for _, row in high_risk_pdf.iterrows():
    evidence = retrieve_contract_evidence(
        "delay penalty milestone SLA breach",
        contract_id=row["contract_id"],
        top_k=1
    )

    hybrid_rows.append({
        "contract_id": row["contract_id"],
        "supplier_name": row["supplier_name"],
        "next_milestone": row["next_milestone"],
        "risk_type": row["risk_type"],
        "recommended_action": row["recommended_action"],
        "supporting_contract_evidence": (
            evidence.iloc[0]["clause_text"]
            if not evidence.empty
            else "No evidence retrieved"
        )
    })

hybrid_pdf = pd.DataFrame(hybrid_rows)

display(hybrid_pdf)


## Section 12 — Agent-Ready Foundation

### Structured tool
```python
query_contract_data(
    intent="high_risk | expiring | payments_due | supplier_summary",
    days=90,
    supplier_id=None
)
```

### Evidence tool
```python
retrieve_contract_evidence(
    query="...",
    contract_id=None,
    top_k=5
)
```

### Next agentic cycle

```text
Observe
  │
  ▼
Plan
  │
  ├── structured facts → query_contract_data(...)
  ├── contract evidence → retrieve_contract_evidence(...)
  │
  ▼
Reason about risk / obligations / payments
  │
  ▼
Propose follow-up action
  │
  ▼
Human approval
  │
  ▼
Action queue
  │
  ▼
Trace and evaluate
```


In [0]:
# Print the final POC assets and interfaces that are ready for the later agentic contract notebook.
ready_objects = {
    "supplier_master": SUPPLIER_TABLE,
    "contract_master": CONTRACT_TABLE,
    "contract_documents": DOCUMENT_TABLE,
    "ariba_metadata": ARIBA_TABLE,
    "erp_payments": ERP_TABLE,
    "sharepoint_metadata": SHAREPOINT_TABLE,
    "structured_extractions": EXTRACTION_TABLE,
    "obligations": OBLIGATION_TABLE,
    "milestones": MILESTONE_TABLE,
    "payments": PAYMENT_TABLE,
    "risks": RISK_TABLE,
    "clause_corpus": CLAUSE_TABLE,
    "enterprise_gold": GOLD_TABLE,
    "enterprise_contract_view": VIEW_NAME,
    "materialized_view_created": MATERIALIZED_VIEW_CREATED,
    "extraction_mode": EXTRACTION_MODE,
    "structured_tool": "query_contract_data",
    "retrieval_tool": "retrieve_contract_evidence"
}

print(json.dumps(ready_objects, indent=2))


## Completion Checklist

- Synthetic supplier master created
- Ariba metadata created
- ERP payment schedule created
- SharePoint / repository metadata created
- Contract corpus created
- `ai_query` extraction attempted
- Deterministic fallback included
- Raw extraction JSON retained
- Obligations normalized
- Milestones normalized
- Payments normalized
- Risks normalized
- Data-quality checks included
- Synthetic ground truth retained for evaluation
- Clause chunks created
- Local retrieval implemented
- AI Search-ready Delta table created
- Enterprise Gold table created
- Materialized view attempted
- Standard-view fallback included
- Structured query tool created
- Evidence retrieval tool created
- Hybrid intelligence example included

### Next notebook
**Lab B — Agentic Contract Intelligence**

`Supervisor → Contract/Risk reasoning → Follow-up recommendation → Human approval → Action queue → MLflow tracing/evaluation`
